In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

In [0]:
%run ./common

In [0]:
def check_nulls(df: DataFrame, cols: list, table: str) -> dict:
    total = df.count()
    result = {}
    for col in cols:
        nulls = df.filter(F.col(col).isNull()).count()
        pct = round(nulls / total * 100, 2) if total else 0
        result[col] = {"nulls": nulls, "pct": pct}
        if pct > 10:
            print(f"  [WARN] {table}.{col}: {pct}% nulos (umbral 10%)")
    return result


In [0]:
def check_duplicates(df: DataFrame, pk_cols: list, table: str) -> int:
    dupes = df.groupBy(pk_cols).count().filter(F.col("count") > 1).count()
    if dupes:
        print(f"  [WARN] {table}: {dupes} claves duplicadas en {pk_cols}")
    return dupes

In [0]:
def check_referential_integrity(df_fact: DataFrame, df_dim: DataFrame, fk_col: str, pk_col: str, table: str) -> DataFrame:
    # Registros huérfanos (FK sin match en dimensión)
    orphans = df_fact.join(df_dim.select(pk_col), df_fact[fk_col] == df_dim[pk_col], "left_anti")
    if orphans.count():
        print(f"  [WARN] {table}.{fk_col}: {orphans} registros huérfanos")
    return orphans

In [0]:
def check_negative_values(df: DataFrame, cols: list, table: str) -> int:
    # Valores negativos en columnas que deben ser >= 0
    total = 0
    for col in cols:
        neg = df.filter(F.col(col) < 0).count()
        if neg:
            print(f"  [WARN] {table}.{col}: {neg} valores negativos")
        total += neg
    return total

In [0]:
def quality_report(df: DataFrame, table: str, pk_cols: list, nullable_cols: list, non_negative_cols: list) -> dict:
    print(f"\n--- Calidad: {table} ---")
    total = df.count()
    report = {
        "table": table,
        "ts": now_utc(),
        "total_records": total,
        "null_check": check_nulls(df, nullable_cols, table),
        "duplicate_pks": check_duplicates(df, pk_cols, table),
        "negative_values": check_negative_values(df, non_negative_cols, table),
    }
    compliant = total - report["duplicate_pks"] - report["negative_values"]
    report["pct_compliant"] = round(compliant / total * 100, 2) if total else 0
    print(f"  Conformes: {report['pct_compliant']}%")
    return report